# L06 · 统计基础 + 交易成本

**预计时长**：60 min | **难度**：⭐⭐⭐ | **前置**：L05

## 本节目标
1. 均值 / 方差 / 标准差 / 相关系数 的金融含义
2. A 股交易成本三件套：佣金 / 印花税 / 过户费
3. 写一个 `apply_trading_cost` 函数，回测必备
4. seaborn 相关矩阵热力图

## 第 1 段：金融概念

### 基础统计的金融含义
| 统计量 | 公式 | 金融含义 |
|-------|------|---------|
| 均值 μ | Σx / N | 平均日收益（"赚钱能力"） |
| 方差 σ² | Σ(x-μ)² / N | 收益波动幅度 |
| 标准差 σ | √方差 | **风险**的量化 |
| 相关系数 ρ | cov(x,y) / (σx·σy) | 两资产同向度（-1 到 +1） |
| 协方差 | E[(x-μx)(y-μy)] | 相关系数的分子 |

### 关键概念：风险与收益的权衡
- **夏普比率 = (μ - rf) / σ**：单位风险的超额回报（rf = 无风险利率，A 股常用 3%）
- 夏普 > 1：不错；> 2：优秀；> 3：可疑（要排查未来函数）

### A 股交易成本三件套（2026 现行）
| 项目 | 费率 | 收费方 | 双向？ |
|------|------|-------|--------|
| 佣金 | 行业平均 **万 2.1**（券商可谈到 **万 1 免 5**） | 券商 | 双向 |
| 印花税 | **卖**方 0.05%（2023.8.28 起） | 国家 | 仅卖出 |
| 过户费 | 0.001%（沪深统一，2022.4.29 起） | 中登公司 | 双向 |
| ~~最低佣金 5 元~~ | 部分券商已"免 5"（不足 5 元按实际收） | 券商 | 双向 |

### 滑点（slippage）
你下单时看到的价格 ≠ 实际成交价。原因：盘口变化、流动性不足。
- 大盘股：滑点极小（< 0.05%）
- 小盘股：滑点可能 0.1–0.3%
- 涨停封板：你可能根本买不到（"一字板"）

回测一般保守按 **0.1% 滑点** 额外扣除。

### 单边 vs 双边总成本（关键数字，以万 2.5 佣金为保守估计）
- **买入单边**：佣金 0.025% + 过户费 0.001% + 滑点 0.1% = **约 0.13%**
- **卖出单边**：上面 + 印花税 0.05% = **约 0.18%**（卖出多一道税）
- **双边来回**：0.13% + 0.18% = **约 0.31%**
- 高频策略每多 1 次调仓，先扣 0.31% 再算收益。**这是策略亏钱的常见原因**。
- 若你的券商给到"万 1 免 5"（2026 主流），单边佣金可降到 0.01%，双边总成本降到 **约 0.25%**。
- `apply_trading_cost` 默认用万 2.5 + 最低 5 元（保守），未实现"免 5"实际。

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation 目录 + project root（兼容两种 jupyter 启动位置）
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (_cwd / 'learning' / 'phase1_foundation')
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data

## 第 2 段：基础统计量计算

In [ ]:
byd = get_stock_data('002594').set_index('date')
rets = byd['close'].pct_change().dropna()

print("比亚迪日收益率基础统计：")
print(f"  均值（日均）:   {rets.mean()*100:>7.3f}%")
print(f"  标准差（日波动）: {rets.std()*100:>7.3f}%")
print(f"  方差:           {rets.var()*1e6:>7.1f}（×1e6）")
print(f"  年化均值:       {rets.mean()*252*100:>7.2f}%")
print(f"  年化波动:       {rets.std()*np.sqrt(252)*100:>7.2f}%")
# 2026 年中国 10 年期国债 ~1.75%，老教材常用 3% 偏高
rf = 0.0175
print(f"  夏普（rf={rf*100:.2f}%）:  {(rets.mean()*252 - rf) / (rets.std()*np.sqrt(252)):>7.2f}")

## 第 3 段：相关矩阵 + 热力图

In [ ]:
import seaborn as sns

# 三股对齐的收益率（用 L04 的 panel）
from pathlib import Path
panel_path = Path("data/panel_three_stocks.parquet")
if panel_path.exists():
    wide = pd.read_parquet(panel_path)
else:
    # 兜底：现场对齐
    frames = {c: get_stock_data(c).set_index('date')['close'] for c in ['002594','002602','002624']}
    wide = pd.DataFrame(frames).sort_index().ffill()

rets_wide = wide.pct_change().dropna()
corr = rets_wide.corr()
print("相关矩阵：")
print(corr.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap='RdYlGn_r', vmin=-1, vmax=1,
            square=True, ax=ax, fmt='.2f')
ax.set_title("三股日收益率相关矩阵")
plt.tight_layout(); plt.show()

**解读**：
- 相关系数接近 1：两股高度同向（行业相近、大盘股）
- 接近 0：相互独立
- 接近 -1：反向（很少见，可能套利对）
- 三只消费/游戏股彼此相关系数通常 0.4–0.7

## 第 4 段：交易成本函数

In [ ]:
def apply_trading_cost(turnover_ratio: float, capital: float,
                       commission_bps: float = 2.5,
                       stamp_duty_bps: float = 5,
                       transfer_fee_bps: float = 0.1,
                       slippage_bps: float = 10) -> float:
    """计算单期交易成本（人民币）。

    Args:
        turnover_ratio: 换手率（0~1 之间，1 = 全仓切换）
        capital: 当期总资金
        *_bps: 各项费率，单位万分之一（bps = basis points）

    Returns:
        本期交易成本（元）
    """
    traded_value = capital * turnover_ratio
    # 买入：佣金 + 过户费 + 滑点
    # 卖出：佣金 + 过户费 + 印花税 + 滑点
    # 双边总费率（万分之）
    total_bps = (commission_bps + transfer_fee_bps + slippage_bps) * 2 \
                + stamp_duty_bps
    return traded_value * total_bps / 10000

# 示例：10 万资金，本期换手 50%（半仓切换）
cost = apply_trading_cost(turnover_ratio=0.5, capital=100_000)
print(f"10 万资金半仓切换成本: {cost:.2f} 元 ({cost/100_000*100:.3f}%)")

## 第 5 段：含成本 vs 不含成本对比

In [ ]:
# 简单策略：每月调仓一次（22 交易日），全仓持有比亚迪
byd = get_stock_data('002594').set_index('date')
byd['ret'] = byd['close'].pct_change()
byd['cum_no_cost'] = (1 + byd['ret']).cumprod()

# 每月扣一次成本
capital = 1.0
capitals = []
for i, r in enumerate(byd['ret']):
    capital *= (1 + r)
    if i > 0 and i % 22 == 0:  # 每月
        cost = apply_trading_cost(turnover_ratio=1.0, capital=capital)
        capital -= cost
    capitals.append(capital)
byd['cum_with_cost'] = capitals

fig, ax = plt.subplots(figsize=(12, 5))
byd['cum_no_cost'].plot(ax=ax, label='不含成本', linewidth=2)
byd['cum_with_cost'].plot(ax=ax, label='含成本', linestyle='--')
ax.legend()
ax.set_title('比亚迪 月调仓：成本对净值曲线的影响')
plt.show()

final_no = byd['cum_no_cost'].iloc[-1]
final_yes = byd['cum_with_cost'].iloc[-1]
print(f"终值 不含成本: {final_no:.3f}  含成本: {final_yes:.3f}  成本吃掉: {(final_no-final_yes)/final_no*100:.2f}%")

## 第 6 段：随堂小练

### 小练：模拟一次完整买卖扣费
100 万资金，100 元买入 5000 股比亚迪，250 元全部卖出。算实际盈利和收益率（含所有成本）。

In [ ]:
# TODO: 你的代码（约 8 行）
# 1) 买入：100 * 5000 = 50万 成交额，扣 commission + transfer + slippage
# 2) 卖出：250 * 5000 = 125万 成交额，扣 commission + transfer + stamp + slippage
# 3) 净利 = 卖出所得 - 买入花费 - 总成本
# 4) 对比"理想"收益（不含成本）和"实际"收益

## 第 7 段：课后练习 + 下节预告

### 📝 `exercises/ex06.py`
1. 写 `risk_metrics(df, rf=0.03)` 返回 dict 含 mean/std/sharpe/max_drawdown
2. 写 `backtest_with_cost(df, signal, cost_bps=15)` 返回含成本的净值曲线
3. 对三股 + 上证指数（000001）算相关矩阵，找出"和比亚迪相关性最低的"

### 🔮 下节 L07：技术指标入门
MA / EMA / MACD / RSI 原理与实现，并用 `np.where` 生成信号——为 Phase 2 的策略回测铺路。

## 第 8 段：Jupyter tip 🔧
- `np.log(1 + rets)`：对数收益率，加性变可加，统计建模常用
- `pd.Timedelta`、`pd.Timestamp`：比 datetime 模块好用
- `np.sqrt(252)`：年化因子。252 是美股 ADR；A 股实际 244 左右，但学界惯例 252
- `ax.yaxis.set_major_formatter(plt.FuncFormatter(...))`：自定义坐标轴格式（如百分号）